In [ ]:
# ================================
# STEP 0: INSTALL & SETUP
# ================================

import os
import numpy as np
import pandas as pd
import torch

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

from sklearn.metrics import accuracy_score, precision_recall_fscore_support

SEED = 42
torch.manual_seed(SEED)

print("✅ Setup complete")

✅ Setup complete


In [ ]:
# ================================
# STEP 1: LOAD PROCESSED DATA
# ================================
train_df = pd.read_csv("/content/train.csv")
test_df = pd.read_csv("/content/test.csv")

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

display(train_df.head())

Train shape: (102080, 4)
Test shape: (25520, 4)


,text,category,text_length,clean_text
0,Report: Consumers tuning in to plasma TVs Firs...,Sci/Tech,37,report consumers tuning in to plasma tvs first...
1,"ICANN STILL CAN, SAYS COURT A US District Cour...",Sci/Tech,35,icann still can says court a us district court...
2,Sirius to Air Men's NCCA Tournament (AP) AP - ...,Sports,37,sirius to air mens ncca tournament ap ap siriu...
3,US-Led Forces Thrust Towards Central Falluja U...,World,38,usled forces thrust towards central falluja us...
4,"Stocks Fall, Led by Tech Sector Technology sto...",Business,32,stocks fall led by tech sector technology stoc...


In [ ]:
# ================================
# STEP 2: LABEL ENCODING
# ================================

label2id = {label: idx for idx, label in enumerate(train_df["category"].unique())}
id2label = {idx: label for label, idx in label2id.items()}

train_df["label"] = train_df["category"].map(label2id)
test_df["label"] = test_df["category"].map(label2id)

num_labels = len(label2id)

print("Label mapping:", label2id)

Label mapping: {'Sci/Tech': 0, 'Sports': 1, 'World': 2, 'Business': 3}


In [ ]:
# ================================
# STEP 3: CONVERT TO HF DATASET
# ================================

train_dataset = Dataset.from_pandas(train_df[["text", "label"]])
test_dataset = Dataset.from_pandas(test_df[["text", "label"]])

print("✅ Converted to HF Dataset")

✅ Converted to HF Dataset


In [ ]:
# ================================
# STEP 4: TOKENIZATION
# ================================

MODEL_NAME = "bert-base-uncased"
MAX_LEN = 64

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_function(example):
    return tokenizer(
        example["text"],
        padding="max_length",
        truncation=True,
        max_length=MAX_LEN
    )

train_dataset = train_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

print("✅ Tokenization complete")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/102080 [00:00<?, ? examples/s]

Map:   0%|          | 0/25520 [00:00<?, ? examples/s]

✅ Tokenization complete


In [ ]:
# ================================
# STEP 5: FORMAT DATASET
# ================================

train_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "label"]
)

test_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "label"]
)

print("✅ Dataset formatted for PyTorch")

✅ Dataset formatted for PyTorch


In [ ]:
# ================================
# STEP 6: LOAD Bert MODEL(Teacher Model)
# ================================

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

print(f"✅ Model loaded on {device}")

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


✅ Model loaded on cuda


In [ ]:
# ================================
# STEP 7: METRICS
# ================================

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, predictions, average='weighted'
    )

    acc = accuracy_score(labels, predictions)

    return {
        "accuracy": acc,
        "f1": f1,
        "precision": precision,
        "recall": recall
    }

In [ ]:
# ================================
# STEP 8: TRAINING ARGUMENTS
# ================================

import os
os.environ["TENSORBOARD_LOGGING_DIR"] = "./logs"

training_args = TrainingArguments(
    output_dir="./results",

    eval_strategy="epoch",
    save_strategy="epoch",

    logging_strategy="steps",
    logging_steps=100,

    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,

    num_train_epochs=3,
    weight_decay=0.01,

    load_best_model_at_end=True,
    metric_for_best_model="f1",

    save_total_limit=2,
    report_to="none"
)

In [ ]:
# ================================
# STEP 9: TRAINER SETUP
# ================================

from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    processing_class=tokenizer,   # ✅ replaces tokenizer in v5
    compute_metrics=compute_metrics
)

print("✅ Trainer initialized")

✅ Trainer initialized


In [ ]:
# ================================
# STEP 10: TRAIN MODEL
# ================================
# --- Setup ---
import gc
gc.collect()
torch.cuda.empty_cache()

trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.205099,0.192647,0.941732,0.941692,0.942056,0.941732
2,0.169453,0.200587,0.946552,0.946438,0.946459,0.946552
3,0.078581,0.227465,0.946944,0.946949,0.946972,0.946944


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=19140, training_loss=0.15445043330033123, metrics={'train_runtime': 499.0432, 'train_samples_per_second': 613.654, 'train_steps_per_second': 38.353, 'total_flos': 1.007207206207488e+16, 'train_loss': 0.15445043330033123, 'epoch': 3.0})

In [ ]:
# ================================
# STEP 11: EVALUATION
# ================================

results = trainer.evaluate()
print("Evaluation Results:", results)

Evaluation Results: {'eval_loss': 0.22746524214744568, 'eval_accuracy': 0.9469435736677116, 'eval_f1': 0.9469489972555946, 'eval_precision': 0.9469724092592381, 'eval_recall': 0.9469435736677116, 'eval_runtime': 13.7008, 'eval_samples_per_second': 1862.67, 'eval_steps_per_second': 116.417, 'epoch': 3.0}


In [ ]:
# ================================
# STEP 12: SAVE MODEL
# ================================

MODEL_PATH = "bert_teacher_model"

trainer.save_model(MODEL_PATH)
tokenizer.save_pretrained(MODEL_PATH)

print("✅ Teacher model saved")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Teacher model saved


In [ ]:
!zip -r bert_teacher_model.zip bert_teacher_model

  adding: bert_teacher_model/ (stored 0%)
  adding: bert_teacher_model/training_args.bin (deflated 53%)
  adding: bert_teacher_model/config.json (deflated 54%)
  adding: bert_teacher_model/tokenizer_config.json (deflated 42%)
  adding: bert_teacher_model/tokenizer.json (deflated 71%)
  adding: bert_teacher_model/model.safetensors (deflated 7%)
